# Does the label-free assessor actually work?

The assessor ranks models without ground-truth labels. That is only useful if the ranking is
correct. This notebook checks it the honest way: on data where we *do* know the answer, we
measure each model's true accuracy and ask whether the label-free score would have picked the
same winner.

We build the labeled data from the same district mapping used earlier. Each historical name
has a known modern (2001) name, so every (old name -> modern name) pair is a labeled example.
Everything runs offline.

In [1]:
import pandas as pd
from importlib.resources import files

from alethia.assess import LabeledDataset, validate_assessor

mapping = pd.read_csv(files("alethia.data") / "India_district_mappings.csv")
references = sorted(mapping["2001"].dropna().str.strip().unique())

def labeled_from(decade):
    pairs = mapping[[decade, "2001"]].dropna()
    pairs[decade] = pairs[decade].str.strip()
    pairs["2001"] = pairs["2001"].str.strip()
    pairs = pairs[pairs[decade] != pairs["2001"]].drop_duplicates(decade)
    return LabeledDataset(
        name=f"{decade}_to_2001",
        queries=pairs[decade].tolist(),
        references=references,
        truth=pairs["2001"].tolist(),
    )

datasets = [labeled_from("1951"), labeled_from("1971"), labeled_from("1991")]
for d in datasets:
    print(f"{d.name}: {len(d.queries)} labeled renames")

1951_to_2001: 190 labeled renames
1971_to_2001: 183 labeled renames
1991_to_2001: 130 labeled renames


## Run the meta-evaluation

For each model and dataset, `validate_assessor` measures true top-1 accuracy and computes the
label-free score, then correlates the two rankings.

In [2]:
result = validate_assessor(
    datasets,
    models={
        "MiniLM": "all-MiniLM-L6-v2",
        "mpnet": "all-mpnet-base-v2",
    },
    metric="top1",
)
result.to_table()

,dataset,model,true_top1,true_mrr,label_free_score
0,1951_to_2001,MiniLM,0.115789,0.152221,5.75
1,1951_to_2001,mpnet,0.105263,0.154229,-5.75
2,1971_to_2001,MiniLM,0.071038,0.113515,4.25
3,1971_to_2001,mpnet,0.065574,0.115076,-4.25
4,1991_to_2001,MiniLM,0.123077,0.144763,4.25
5,1991_to_2001,mpnet,0.092308,0.125780,-4.25


In [3]:
print(f"Kendall tau (predicted vs true ranking): {result.kendall_tau:.3f}")
print(f"Spearman rho: {result.spearman_rho:.3f}")

Kendall tau (predicted vs true ranking): 1.000
Spearman rho: 1.000


A positive correlation means the label-free score ranks the models the same way their
measured accuracy does. That is the evidence that justifies using the assessor on a new
dataset where you have no labels at all.

These historical renames are deliberately hard: a name like "Balipara Frontier Tract" shares
almost no surface form with its modern equivalent, so absolute accuracy is low. The point of
this notebook is the *ranking*, not the absolute numbers.

## Takeaways

- The assessor's label-free verdict can be checked against ground truth wherever labels exist.
- On these district renames its ranking agrees with measured top-1 accuracy.
- Run `validate_assessor` on your own labeled subset to calibrate trust before relying on the
  label-free score for the unlabeled rest of your data.